In [2]:
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

{'A': 2, 'B': 1}


{'A': 2, 'B': 1}
.get(c, 0) при первой встрече буквы возвращает 0, а при повторной — уже накопленное значение. Буква “A” встретилась дважды, поэтому её счётчик вырос до 2

In [3]:
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

[0, 1, 1]


[0, 1, 1]
список передаётся в функцию по ссылке, а не копируется. x внутри функции — тот же объект, что и a снаружи, поэтому оба вызова .append(1) добавляют элементы в один и тот же список

In [4]:
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

2


2

.lower() делает "Almaty" и "ALMATY" одинаковой строкой ("almaty"), а set не хранит повторяющиеся значения — из трёх строк осталось два уникальных элемента

In [5]:
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

{'ab': 2, 'abc': 3}


{'ab': 2, 'abc': 3}
ключи словаря уникальны — повторный ключ "ab" не добавляется отдельно, а перезаписывает значение по тому же ключу

In [6]:
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

5
None


сначала выведется 5, потом None
print(n) просто печатает значение на экран — это не то же самое, что return. Функция без return автоматически возвращает None, поэтому именно None попадает в x и выводится вторым print(x)

In [7]:
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [8]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


In [9]:
def to_num(s):
    return float(s.replace(" ", ""))

def clean_city(s):
    s = s.strip().lower()
    mapping = {
        "алматы": "Алматы",
        "almaty": "Алматы",
        "астана": "Астана",
        "astana": "Астана",
        "шымкент": "Шымкент",
        "shymkent": "Шымкент",
        "караганда": "Караганда",
        "актобе": "Актобе",
        "атырау": "Атырау",
    }
    return mapping.get(s, s)

In [11]:
category_count = {}
for r in rows:
    cat = r["category"]
    category_count[cat] = category_count.get(cat, 0) + 1


for cat, count in sorted(category_count.items(), key=lambda x: x[1], reverse=True):
    print(cat, count)

Одежда 1529
Электроника 1185
Дом 1105
Красота 991
Книги 653


In [12]:
revenue = {}
count_paid = {}

for r in rows:
    if r["status"] == "оплачен":
        cat = r["category"]
        amt = to_num(r["amount"])
        revenue[cat] = revenue.get(cat, 0) + amt
        count_paid[cat] = count_paid.get(cat, 0) + 1

for cat in revenue:
    avg_check = revenue[cat] / count_paid[cat]
    print(f"{cat} | выручка: {revenue[cat]:.0f} | средний чек: {avg_check:.0f}")

Одежда | выручка: 18321080 | средний чек: 13286
Красота | выручка: 5507620 | средний чек: 6188
Дом | выручка: 10516320 | средний чек: 10655
Книги | выручка: 2294100 | средний чек: 3935
Электроника | выручка: 65270240 | средний чек: 61287


In [13]:
month_count = {}
for r in rows:
    month = r["order_date"][:7]
    month_count[month] = month_count.get(month, 0) + 1

for month in sorted(month_count):
    print(month, month_count[month])

2026-09 2782
2026-10 2681


In [14]:
month_revenue = {}
for r in rows:
    if r["status"] == "оплачен":
        month = r["order_date"][:7]
        month_revenue[month] = month_revenue.get(month, 0) + to_num(r["amount"])

for month in sorted(month_revenue):
    print(month, month_revenue[month])

sep = month_revenue.get("2026-09", 0)
oct_ = month_revenue.get("2026-10", 0)
change = (oct_ - sep) / sep * 100
print(f"Изменение октября к сентябрю: {change:.1f}%")

2026-09 57489850.0
2026-10 44419510.0
Изменение октября к сентябрю: -22.7%


In [15]:
total_by_city = {}
cancelled_by_city = {}

for r in rows:
    city = clean_city(r["city"])
    total_by_city[city] = total_by_city.get(city, 0) + 1
    if r["status"] == "отменён":
        cancelled_by_city[city] = cancelled_by_city.get(city, 0) + 1

for city in total_by_city:
    cancelled = cancelled_by_city.get(city, 0)
    share = cancelled / total_by_city[city] * 100
    print(f"{city}: {share:.1f}%")

Алматы: 7.9%
Астана: 8.8%
Караганда: 8.9%
Шымкент: 20.4%
Атырау: 7.5%
Актобе: 10.3%


In [16]:
revenue_by_customer = {}
for r in rows:
    if r["status"] == "оплачен":
        cid = r["customer_id"]
        revenue_by_customer[cid] = revenue_by_customer.get(cid, 0) + to_num(r["amount"])

top5 = sorted(revenue_by_customer.items(), key=lambda x: x[1], reverse=True)[:5]
for cid, rev in top5:
    print(cid, rev)

TEST 29999970.0
C01627 257180.0
C00485 215920.0
C01338 192080.0
C02122 191670.0


In [17]:
customers_by_city = {}
for r in rows:
    city = clean_city(r["city"])
    if city not in customers_by_city:
        customers_by_city[city] = set()
    customers_by_city[city].add(r["customer_id"])

for city, customers in customers_by_city.items():
    print(city, len(customers))

Алматы 1351
Астана 981
Караганда 451
Шымкент 655
Атырау 138
Актобе 358
